<a href="https://colab.research.google.com/github/aarushi-202618017/202618017_Aarushi_DS605/blob/main/202618017_LAB_06/202618017_LAB_06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Part A - Image Feature Extraction and Classification**

In [ ]:
! mkdir dataset
! mkdir dataset/cracks
! mkdir dataset/noncracks

In [ ]:
import os

folder_path = '/content/dataset/noncracks'
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.gif')
image_count = sum(
    1
    for f in os.listdir(folder_path)
    if f.lower().endswith(valid_extensions)
)
print(f'Total images: {image_count}')


Total images: 200


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
img = cv2.imread('/content/dataset/cracks/IMG_20180514_165251954_HDR resized_448.jpg')

In [ ]:
img.shape

(448, 448, 3)

In [ ]:
import os
import cv2

dataset_folders = {
    'noncracks': 0,
    'cracks': 1
}

base_dir = "/content/dataset"

processed_images = []
labels = []

for folder_name, label in dataset_folders.items():
    folder_path = os.path.join(base_dir, folder_name)

    if not os.path.exists(folder_path):
        print(f"not found {folder_path}")
        continue

    for file_name in os.listdir(folder_path):
        if file_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            file_path = os.path.join(folder_path, file_name)

            img = cv2.imread(file_path)
            if img is None:
                continue

            img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

            processed_images.append(img_gray)
            labels.append(label)

print(f"processed {len(processed_images)} total images.")
print(f"Non-Crack count: {labels.count(0)} | Crack count: {labels.count(1)}")

processed 400 total images.
Non-Crack count: 200 | Crack count: 200


In [ ]:
def extract_features(img_gray):
    total_pixels = img_gray.size
    mean_val = np.mean(img_gray)
    std_val = np.std(img_gray)
    dark_ratio = np.sum(img_gray < 50) / total_pixels
    bright_ratio = np.sum(img_gray > 200) / total_pixels
    median_val = np.median(img_gray)
    min_val = np.min(img_gray)
    max_val = np.max(img_gray)

    edges = cv2.Canny(img_gray, threshold1=50, threshold2=150)
    edge_count = np.sum(edges > 0)
    edge_density = edge_count / total_pixels

    return [
        mean_val, std_val, dark_ratio, bright_ratio,
        median_val, min_val, max_val, edge_count, edge_density
    ]

feature_matrix = [extract_features(img) for img in processed_images]

feature_names = [
    'mean_brightness', 'contrast_std', 'dark_pixel_ratio', 'bright_pixel_ratio',
    'median_intensity', 'min_intensity', 'max_intensity', 'edge_count', 'edge_density'
]

X_df = pd.DataFrame(feature_matrix, columns=feature_names)
X = X_df.values
y = np.array(labels)

print(f"Feature matrix shape: {X.shape}")
print(f"Target array shape: {y.shape}")
print("\nFirst row preview:")
print(X_df.head(1).T)

Feature matrix shape: (400, 9)
Target array shape: (400,)

First row preview:
                               0
mean_brightness       157.305619
contrast_std           18.777875
dark_pixel_ratio        0.000000
bright_pixel_ratio      0.011853
median_intensity      157.000000
min_intensity          70.000000
max_intensity         255.000000
edge_count          70213.000000
edge_density            0.349834


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train shape: {X_train_scaled.shape}  | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test_scaled.shape}   | y_test shape:  {y_test.shape}")

X_train shape: (320, 9)  | y_train shape: (320,)
X_test shape:  (80, 9)   | y_test shape:  (80,)


In [ ]:
import time
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

lr_model = LogisticRegression(random_state=42)

t0_train = time.time()
lr_model.fit(X_train_scaled, y_train)
train_time_lr = time.time() - t0_train

t0_pred = time.time()
y_pred_lr = lr_model.predict(X_test_scaled)
pred_time_lr = time.time() - t0_pred

cm_lr = confusion_matrix(y_test, y_pred_lr)
lr_results = pd.DataFrame([{
    'Model': 'Logistic Regression',
    'Accuracy': round(accuracy_score(y_test, y_pred_lr), 4),
    'Precision': round(precision_score(y_test, y_pred_lr, zero_division=0), 4),
    'Recall': round(recall_score(y_test, y_pred_lr, zero_division=0), 4),
    'F1-Score': round(f1_score(y_test, y_pred_lr, zero_division=0), 4),
    'Train Time (s)': round(train_time_lr, 5),
    'Pred Time (s)': round(pred_time_lr, 5),
    'Confusion Matrix [TN, FP, FN, TP]': cm_lr.ravel().tolist()
}])

lr_results

,Model,Accuracy,Precision,Recall,F1-Score,Train Time (s),Pred Time (s),"Confusion Matrix [TN, FP, FN, TP]"
0,Logistic Regression,0.8625,0.8919,0.825,0.8571,0.01312,0.00048,"[36, 4, 7, 33]"


In [ ]:
import time
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

t0_train = time.time()
rf_model.fit(X_train_scaled, y_train)
train_time_rf = time.time() - t0_train

t0_pred = time.time()
y_pred_rf = rf_model.predict(X_test_scaled)
pred_time_rf = time.time() - t0_pred

cm_rf = confusion_matrix(y_test, y_pred_rf)
rf_results = pd.DataFrame([{
    'Model': 'Random Forest',
    'Accuracy': round(accuracy_score(y_test, y_pred_rf), 4),
    'Precision': round(precision_score(y_test, y_pred_rf, zero_division=0), 4),
    'Recall': round(recall_score(y_test, y_pred_rf, zero_division=0), 4),
    'F1-Score': round(f1_score(y_test, y_pred_rf, zero_division=0), 4),
    'Train Time (s)': round(train_time_rf, 5),
    'Pred Time (s)': round(pred_time_rf, 5),
    'Confusion Matrix [TN, FP, FN, TP]': cm_rf.ravel().tolist()
}])

rf_results

,Model,Accuracy,Precision,Recall,F1-Score,Train Time (s),Pred Time (s),"Confusion Matrix [TN, FP, FN, TP]"
0,Random Forest,0.9625,0.9512,0.975,0.963,0.26499,0.00942,"[38, 2, 1, 39]"


# **Part B - Text Vectorization and Spam Classification**

In [12]:
import kagglehub

path = kagglehub.dataset_download("balaka18/email-spam-classification-dataset-csv")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'email-spam-classification-dataset-csv' dataset.
Path to dataset files: /kaggle/input/email-spam-classification-dataset-csv


In [13]:
import numpy as np
import pandas as pd

In [14]:
email=pd.read_csv("/kaggle/input/email-spam-classification-dataset-csv/emails.csv")

In [ ]:
email.head()

,Email No.,the,to,ect,and,for,of,a,you,hou,...,connevey,jay,valued,lay,infrastructure,military,allowing,ff,dry,Prediction
0,Email 1,0,0,1,0,0,0,2,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Email 2,8,13,24,6,6,2,102,1,27,...,0,0,0,0,0,0,0,1,0,0
2,Email 3,0,0,1,0,0,0,8,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Email 4,0,5,22,0,5,1,51,2,10,...,0,0,0,0,0,0,0,0,0,0
4,Email 5,7,6,17,1,5,2,57,0,9,...,0,0,0,0,0,0,0,1,0,0


In [ ]:
email.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5172 entries, 0 to 5171
Columns: 3002 entries, Email No. to Prediction
dtypes: int64(3001), object(1)
memory usage: 118.5+ MB


In [17]:
email.isna().sum().value_counts()

,count
0,3002


In [ ]:
email.count()

,0
Email No.,5172
the,5172
to,5172
ect,5172
and,5172
...,...
military,5172
allowing,5172
ff,5172
dry,5172


In [ ]:
email['Prediction'].value_counts()

,count
Prediction,
0,3672
1,1500


In [ ]:
from sklearn.model_selection import train_test_split

X = email.drop(columns=['Email No.', 'Prediction'])
y = email['Prediction']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Number of generated features: {X.shape[1]}")
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape:  {X_test.shape}")

Number of generated features: 3000
Training set shape: (4137, 3000)
Testing set shape:  (1035, 3000)


In [ ]:
X_test

,the,to,ect,and,for,of,a,you,hou,in,...,enhancements,connevey,jay,valued,lay,infrastructure,military,allowing,ff,dry
767,5,2,3,0,2,4,28,1,2,9,...,0,0,0,0,0,0,0,0,1,0
2650,2,5,4,5,3,2,44,3,2,5,...,0,0,0,0,0,0,0,0,1,0
2568,6,3,1,0,2,3,18,0,0,4,...,0,0,0,0,0,0,0,0,0,0
3325,2,2,1,2,1,1,33,2,0,1,...,0,0,0,0,1,0,0,0,1,0
5022,5,19,4,13,5,14,183,9,2,26,...,0,0,0,0,0,0,0,0,12,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1901,7,4,21,6,6,0,79,2,11,8,...,0,0,0,0,0,0,0,0,0,0
1360,0,0,1,0,1,0,2,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1910,1,2,1,0,0,2,6,1,0,1,...,0,0,0,0,0,0,0,0,2,0
1847,1,4,2,0,1,0,9,0,0,2,...,0,0,0,0,0,0,0,0,1,0


In [ ]:
y_test

,Prediction
767,0
2650,0
2568,0
3325,0
5022,0
...,...
1901,0
1360,0
1910,1
1847,0


In [ ]:
import time
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

lr_spam_model = LogisticRegression(max_iter=1000, random_state=42)

t0_train = time.time()
lr_spam_model.fit(X_train, y_train)
train_time_lr = time.time() - t0_train

t0_pred = time.time()
y_pred_lr = lr_spam_model.predict(X_test)
pred_time_lr = time.time() - t0_pred

cm_lr = confusion_matrix(y_test, y_pred_lr)
lr_results = pd.DataFrame([{
    'Model': 'Logistic Regression (Count)',
    'Features': X_train.shape[1],
    'Accuracy': round(accuracy_score(y_test, y_pred_lr), 4),
    'Precision': round(precision_score(y_test, y_pred_lr, zero_division=0), 4),
    'Recall': round(recall_score(y_test, y_pred_lr, zero_division=0), 4),
    'F1-Score': round(f1_score(y_test, y_pred_lr, zero_division=0), 4),
    'Train Time (s)': round(train_time_lr, 5),
    'Pred Time (s)': round(pred_time_lr, 5),
    'Confusion Matrix [TN, FP, FN, TP]': cm_lr.ravel().tolist()
}])

lr_results

,Model,Features,Accuracy,Precision,Recall,F1-Score,Train Time (s),Pred Time (s),"Confusion Matrix [TN, FP, FN, TP]"
0,Logistic Regression (Count),3000,0.9826,0.9578,0.9833,0.9704,17.77407,0.13162,"[722, 13, 5, 295]"


In [ ]:
import time
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

rf_spam_model = RandomForestClassifier(n_estimators=100, random_state=42)

t0_train = time.time()
rf_spam_model.fit(X_train, y_train)
train_time_rf = time.time() - t0_train

t0_pred = time.time()
y_pred_rf = rf_spam_model.predict(X_test)
pred_time_rf = time.time() - t0_pred

cm_rf = confusion_matrix(y_test, y_pred_rf)
rf_results = pd.DataFrame([{
    'Model': 'Random Forest (Count)',
    'Features': X_train.shape[1],
    'Accuracy': round(accuracy_score(y_test, y_pred_rf), 4),
    'Precision': round(precision_score(y_test, y_pred_rf, zero_division=0), 4),
    'Recall': round(recall_score(y_test, y_pred_rf, zero_division=0), 4),
    'F1-Score': round(f1_score(y_test, y_pred_rf, zero_division=0), 4),
    'Train Time (s)': round(train_time_rf, 5),
    'Pred Time (s)': round(pred_time_rf, 5),
    'Confusion Matrix [TN, FP, FN, TP]': cm_rf.ravel().tolist()
}])

rf_results

,Model,Features,Accuracy,Precision,Recall,F1-Score,Train Time (s),Pred Time (s),"Confusion Matrix [TN, FP, FN, TP]"
0,Random Forest (Count),3000,0.9643,0.934,0.9433,0.9386,4.11082,0.05604,"[715, 20, 17, 283]"


# **Part C - Improve the Representation**

In [ ]:
import time
import pandas as pd
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MaxAbsScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

selector = SelectKBest(score_func=chi2, k=1000)
X_train_kbest = selector.fit_transform(X_train, y_train)
X_test_kbest = selector.transform(X_test)

scaler = MaxAbsScaler()
X_train_scaled = scaler.fit_transform(X_train_kbest)
X_test_scaled = scaler.transform(X_test_kbest)

lr_kbest = LogisticRegression(max_iter=1000, random_state=42)

t0_train = time.time()
lr_kbest.fit(X_train_scaled, y_train)
train_time = time.time() - t0_train

t0_pred = time.time()
y_pred_kbest = lr_kbest.predict(X_test_scaled)
pred_time = time.time() - t0_pred

cm = confusion_matrix(y_test, y_pred_kbest)
kbest_results = pd.DataFrame([{
    'Model': 'Logistic Regression (Scaled Top 500 Chi2)',
    'Features': X_train_kbest.shape[1],
    'Accuracy': round(accuracy_score(y_test, y_pred_kbest), 4),
    'Precision': round(precision_score(y_test, y_pred_kbest, zero_division=0), 4),
    'Recall': round(recall_score(y_test, y_pred_kbest, zero_division=0), 4),
    'F1-Score': round(f1_score(y_test, y_pred_kbest, zero_division=0), 4),
    'Train Time (s)': round(train_time, 5),
    'Pred Time (s)': round(pred_time, 5),
    'Confusion Matrix [TN, FP, FN, TP]': cm.ravel().tolist()
}])

kbest_results

,Model,Features,Accuracy,Precision,Recall,F1-Score,Train Time (s),Pred Time (s),"Confusion Matrix [TN, FP, FN, TP]"
0,Logistic Regression (Scaled Top 500 Chi2),1000,0.9556,0.9504,0.8933,0.921,0.42826,0.00583,"[721, 14, 32, 268]"


In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.preprocessing import Normalizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

X_train_log = np.log1p(X_train)
X_test_log = np.log1p(X_test)

normalizer = Normalizer(norm='l2')
X_train_norm = normalizer.fit_transform(X_train_log)
X_test_norm = normalizer.transform(X_test_log)

lr_improved = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

t0_train = time.time()
lr_improved.fit(X_train_norm, y_train)
train_time = time.time() - t0_train

t0_pred = time.time()
y_pred_imp = lr_improved.predict(X_test_norm)
pred_time = time.time() - t0_pred

cm = confusion_matrix(y_test, y_pred_imp)
imp_results = pd.DataFrame([{
    'Model': 'Log-Scaled + L2 Norm + Balanced LR',
    'Features': X_train_norm.shape[1],
    'Accuracy': round(accuracy_score(y_test, y_pred_imp), 4),
    'Precision': round(precision_score(y_test, y_pred_imp, zero_division=0), 4),
    'Recall': round(recall_score(y_test, y_pred_imp, zero_division=0), 4),
    'F1-Score': round(f1_score(y_test, y_pred_imp, zero_division=0), 4),
    'Train Time (s)': round(train_time, 5),
    'Pred Time (s)': round(pred_time, 5),
    'Confusion Matrix [TN, FP, FN, TP]': cm.ravel().tolist()
}])

imp_results

,Model,Features,Accuracy,Precision,Recall,F1-Score,Train Time (s),Pred Time (s),"Confusion Matrix [TN, FP, FN, TP]"
0,Log-Scaled + L2 Norm + Balanced LR,3000,0.944,0.8457,0.9867,0.9108,0.38767,0.00795,"[681, 54, 4, 296]"


- **Best Overall Balanced System:**
  
  Raw Count Logistic Regression (Pipeline 1). It achieves the highest F1-Score (0.9704), highest overall accuracy (98.26%), and keeps legitimate emails out of the spam folder (only 13 FP).


- **Best High-Throughput / Speed System:**

  Scaled 1,000 Chi2 Features (Pipeline 2). It drops feature dimension by 66.7% and executes inference in under 6 milliseconds while retaining 95%+ accuracy and precision.


- **Best Security / Catch-All System:**

  Log-Scaled + Class-Balanced (Pipeline 3). Ideal for threat-hunting environments where missing a single malicious email is unacceptable, achieving a project-high 98.67% Recall.